In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp

## Ejercicio 1

**Problema dual del SVM con margen suave.** Con etiquetas $y_i\in\{-1,1\}$ y la matriz de kernel $K_{ij}=k(x_i,x_j)$:

$$
\begin{array}{ll}
\text{minimizar} & \frac12\lambda^\top P\lambda-\mathbf 1^\top\lambda,\qquad P_{ij}=y_iy_jK_{ij}\\[2pt]
\text{sujeto a} & 0\le\lambda_i\le C,\quad \sum_iy_i\lambda_i=0 .
\end{array}
$$

Es un QP convexo: $P\succeq0$ porque $K\succeq0$. Por redondeo, `cvxpy` puede no reconocer $P$ como semidefinida positiva; `cp.psd_wrap(P)` le indica que lo es. Usamos el solver Clarabel con tolerancias chicas, para que los $\lambda_i$ que deberían valer 0 o $C$ queden muy cerca de esos valores.

**A partir de $\lambda$** (con una tolerancia $\varepsilon$ para comparar con 0 y con $C$):

- **vectores soporte**: las observaciones con $\lambda_i>\varepsilon$; las demás no intervienen en el modelo;
- **sobre el margen**: $\varepsilon<\lambda_i<C-\varepsilon$, que cumplen $y_if(x_i)=1$;
- **en la cota**: $\lambda_i=C$, que están dentro del margen o mal clasificadas ($y_if(x_i)\le1$);
- **función de decisión**: $f(x)=\sum_{i\in SV}\lambda_iy_i\,k(x_i,x)+\beta_0$;
- **bias**: $\beta_0$ es el promedio, sobre los vectores soporte sobre el margen, de $y_i-\sum_j\lambda_jy_jK_{ji}$;
- **kernel lineal**: $f(x)=\beta^\top x+\beta_0$ con $\beta=\sum_i\lambda_iy_ix_i$.

**Kernels**: lineal, $k(x,z)=x^\top z$; RBF, $k(x,z)=\exp(-\gamma\|x-z\|^2)$.

In [ ]:
class mySVM:
    """
    SVM con margen suave, resuelto por el problema dual con cvxpy.
    """
    def __init__(self, C=1.0, kernel="linear", gamma=1.0):
        """
        Inicializa el modelo.

        Parámetros:
        - C: hiperparámetro de regularización
        - kernel: "linear" o "rbf"
        - gamma: parámetro del kernel RBF
        """

        if kernel not in ("linear", "rbf"):
            raise ValueError(f"kernel debe ser 'linear' o 'rbf', no {kernel!r}")
        self.C = C
        self.kernel = kernel
        self.gamma = gamma

        # Resultados del ajuste
        self.lam = None  # multiplicadores de Lagrange (n,)
        self.es_sv = None  # máscara de vectores soporte (n,)
        self.sobre_margen = None  # máscara de vectores soporte sobre el margen (n,)
        self.X_sv = None  # vectores soporte (n_sv, p)
        self.y_sv = None  # sus etiquetas en {-1, 1} (n_sv,)
        self.lam_sv = None  # sus multiplicadores (n_sv,)
        self.beta = None  # (p,), solo para el kernel lineal
        self.beta0 = None

    def matriz_kernel(self, X1, X2):
        """
        Matriz de kernel K_ij = k(x_i, z_j) entre las filas de X1 y las de X2.

        Parámetros:
        - X1: (n1, p)
        - X2: (n2, p)

        Retorna:
        - K: (n1, n2)
        """

        # TODO: lineal, X1 X2^T; RBF, exp(-γ ||x_i - z_j||²) con bucles y vectorizado

        if self.kernel == "linear":
            K = ...  # (n1, n2)
        else:
            # Opción 1: elemento a elemento
            K = np.zeros((X1.shape[0], X2.shape[0]))
            for i in range(X1.shape[0]):
                for j in range(X2.shape[0]):
                    d = ...  # (p,)
                    K[i, j] = ...

            # Opción 2: vectorizada, ||x - z||² = ||x||² + ||z||² - 2 x^T z
            # (n1, 1) + (1, n2) - (n1, n2) -> (n1, n2)
            D = ...
            K = ...
        return K

    def fit(self, X, y):
        """
        Ajusta el modelo resolviendo el problema dual.

        Parámetros:
        - X: (n, p)
        - y: (n,), con valores 0 y 1

        Retorna:
        - self
        """

        # TODO:
        # - Etiquetas en {-1, 1}; matrices K y P
        # - Problema dual con cvxpy (P envuelta con cp.psd_wrap)
        # - Vectores soporte, los que están sobre el margen, β (kernel lineal) y β0

        y = np.where(y == 1, 1.0, -1.0)  # (n,)
        K = self.matriz_kernel(X, X)  # (n, n)

        # Opción 1: elemento a elemento
        P = np.zeros_like(K)  # (n, n)
        for i in range(len(y)):
            for j in range(len(y)):
                P[i, j] = ...

        # Opción 2: vectorizada, (n, 1) * (1, n) * (n, n) -> (n, n)
        P = ...

        lam = cp.Variable(len(y))  # (n,)
        objetivo = cp.Minimize(...)
        restricciones = [
            ...,
            ...,
            ...,
        ]
        problema = cp.Problem(objetivo, restricciones)
        problema.solve(solver=cp.CLARABEL, tol_gap_abs=1e-10, tol_gap_rel=1e-10, tol_feas=1e-10)

        eps = 1e-6 * self.C  # tolerancia para comparar λ con 0 y con C
        lam = np.clip(lam.value, 0, self.C)  # (n,)
        self.lam = lam
        self.es_sv = ...  # (n,)
        self.sobre_margen = ...  # (n,)
        self.X_sv, self.y_sv, self.lam_sv = X[self.es_sv], y[self.es_sv], lam[self.es_sv]

        if self.kernel == "linear":
            # (n_sv,) * (n_sv,) -> (n_sv,); (n_sv,) @ (n_sv, p) -> (p,)
            self.beta = ...

        # β0: promedio de y_i - Σ_j λ_j y_j K_ji sobre los vectores soporte sobre el margen
        # (si ninguno está sobre el margen, aproximamos con todos los vectores soporte)
        h = ...  # Σ_j λ_j y_j K_ij para cada i, (n,)
        S = self.sobre_margen if np.any(self.sobre_margen) else self.es_sv
        self.beta0 = ...
        return self

    def decision_function(self, X):
        """
        f(x) = Σ_{i en SV} λ_i y_i k(x_i, x) + β0.

        Parámetros:
        - X: (m, p)

        Retorna:
        - f: (m,)
        """

        # TODO: kernel entre X y los vectores soporte

        K_sv = ...  # (m, n_sv)
        # (m, n_sv) @ (n_sv,) -> (m,)
        return ...

    def predict(self, X):
        """
        Etiqueta predicha: 1 si f(x) >= 0, 0 si no.

        Parámetros:
        - X: (m, p)

        Retorna:
        - y_pred: (m,)
        """

        # TODO

        return ...  # (m,)

##### Solución

In [ ]:
class mySVM:
    """
    SVM con margen suave, resuelto por el problema dual con cvxpy.
    """
    def __init__(self, C=1.0, kernel="linear", gamma=1.0):
        """
        Inicializa el modelo.

        Parámetros:
        - C: hiperparámetro de regularización
        - kernel: "linear" o "rbf"
        - gamma: parámetro del kernel RBF
        """

        if kernel not in ("linear", "rbf"):
            raise ValueError(f"kernel debe ser 'linear' o 'rbf', no {kernel!r}")
        self.C = C
        self.kernel = kernel
        self.gamma = gamma

        # Resultados del ajuste
        self.lam = None  # multiplicadores de Lagrange (n,)
        self.es_sv = None  # máscara de vectores soporte (n,)
        self.sobre_margen = None  # máscara de vectores soporte sobre el margen (n,)
        self.X_sv = None  # vectores soporte (n_sv, p)
        self.y_sv = None  # sus etiquetas en {-1, 1} (n_sv,)
        self.lam_sv = None  # sus multiplicadores (n_sv,)
        self.beta = None  # (p,), solo para el kernel lineal
        self.beta0 = None

    def matriz_kernel(self, X1, X2):
        """
        Matriz de kernel K_ij = k(x_i, z_j) entre las filas de X1 y las de X2.

        Parámetros:
        - X1: (n1, p)
        - X2: (n2, p)

        Retorna:
        - K: (n1, n2)
        """

        if self.kernel == "linear":
            K = X1 @ X2.T  # (n1, n2)
        else:
            # Opción 1: elemento a elemento
            K = np.zeros((X1.shape[0], X2.shape[0]))
            for i in range(X1.shape[0]):
                for j in range(X2.shape[0]):
                    d = X1[i] - X2[j]  # (p,)
                    K[i, j] = np.exp(-self.gamma * d @ d)

            # Opción 2: vectorizada, ||x - z||² = ||x||² + ||z||² - 2 x^T z
            # (n1, 1) + (1, n2) - (n1, n2) -> (n1, n2)
            D = np.sum(X1**2, axis=1)[:, None] + np.sum(X2**2, axis=1)[None, :] - 2 * X1 @ X2.T
            K = np.exp(-self.gamma * np.maximum(D, 0))
        return K

    def fit(self, X, y):
        """
        Ajusta el modelo resolviendo el problema dual.

        Parámetros:
        - X: (n, p)
        - y: (n,), con valores 0 y 1

        Retorna:
        - self
        """

        y = np.where(y == 1, 1.0, -1.0)  # (n,)
        K = self.matriz_kernel(X, X)  # (n, n)

        # Opción 1: elemento a elemento
        P = np.zeros_like(K)  # (n, n)
        for i in range(len(y)):
            for j in range(len(y)):
                P[i, j] = y[i] * y[j] * K[i, j]

        # Opción 2: vectorizada, (n, 1) * (1, n) * (n, n) -> (n, n)
        P = y[:, None] * y[None, :] * K

        lam = cp.Variable(len(y))  # (n,)
        objetivo = cp.Minimize(0.5 * cp.quad_form(lam, cp.psd_wrap(P)) - cp.sum(lam))
        restricciones = [
            lam >= 0,
            lam <= self.C,
            y @ lam == 0,
        ]
        problema = cp.Problem(objetivo, restricciones)
        problema.solve(solver=cp.CLARABEL, tol_gap_abs=1e-10, tol_gap_rel=1e-10, tol_feas=1e-10)

        eps = 1e-6 * self.C  # tolerancia para comparar λ con 0 y con C
        lam = np.clip(lam.value, 0, self.C)  # (n,)
        self.lam = lam
        self.es_sv = lam > eps  # (n,)
        self.sobre_margen = (lam > eps) & (lam < self.C - eps)  # (n,)
        self.X_sv, self.y_sv, self.lam_sv = X[self.es_sv], y[self.es_sv], lam[self.es_sv]

        if self.kernel == "linear":
            # (n_sv,) * (n_sv,) -> (n_sv,); (n_sv,) @ (n_sv, p) -> (p,)
            self.beta = (self.lam_sv * self.y_sv) @ self.X_sv

        # β0: promedio de y_i - Σ_j λ_j y_j K_ji sobre los vectores soporte sobre el margen
        # (si ninguno está sobre el margen, aproximamos con todos los vectores soporte)
        h = K @ (lam * y)  # Σ_j λ_j y_j K_ij para cada i, (n,)
        S = self.sobre_margen if np.any(self.sobre_margen) else self.es_sv
        self.beta0 = np.mean(y[S] - h[S])
        return self

    def decision_function(self, X):
        """
        f(x) = Σ_{i en SV} λ_i y_i k(x_i, x) + β0.

        Parámetros:
        - X: (m, p)

        Retorna:
        - f: (m,)
        """

        K_sv = self.matriz_kernel(X, self.X_sv)  # (m, n_sv)
        # (m, n_sv) @ (n_sv,) -> (m,)
        return K_sv @ (self.lam_sv * self.y_sv) + self.beta0

    def predict(self, X):
        """
        Etiqueta predicha: 1 si f(x) >= 0, 0 si no.

        Parámetros:
        - X: (m, p)

        Retorna:
        - y_pred: (m,)
        """

        return (self.decision_function(X) >= 0).astype(int)  # (m,)

##### *(Adicional)* Problema primal para el kernel lineal

$$
\begin{array}{ll}
\text{minimizar} & \frac12\|\beta\|^2+C\sum_i\xi_i\\[2pt]
\text{sujeto a} & y_i(\beta^\top x_i+\beta_0)\ge1-\xi_i,\quad \xi_i\ge0 .
\end{array}
$$

En el óptimo, $\xi_i=\max\big(0,\ 1-y_i(\beta^\top x_i+\beta_0)\big)$: es la pérdida *hinge*. Además, los multiplicadores de las restricciones de margen son los $\lambda_i$ del dual.

In [ ]:
def svm_primal(X, y, C):
    """
    SVM lineal con margen suave, resuelto por el problema primal con cvxpy.

    Parámetros:
    - X: (n, p)
    - y: (n,), con valores 0 y 1
    - C: hiperparámetro de regularización

    Retorna:
    - beta: (p,)
    - beta0: float
    - lam: multiplicadores de las restricciones de margen (n,)
    """

    # TODO: variables β, β0 y ξ; restricciones de margen y ξ >= 0; los multiplicadores con dual_value

    y = np.where(y == 1, 1.0, -1.0)  # (n,)
    n, p = X.shape
    beta, beta0, xi = ..., ..., ...
    margen = ...  # restricción y_i (β^T x_i + β0) >= 1 - ξ_i
    problema = cp.Problem(..., [margen, ...])
    problema.solve(solver=cp.CLARABEL, tol_gap_abs=1e-10, tol_gap_rel=1e-10, tol_feas=1e-10)
    return beta.value, beta0.value, margen.dual_value

##### Solución

In [ ]:
def svm_primal(X, y, C):
    """
    SVM lineal con margen suave, resuelto por el problema primal con cvxpy.

    Parámetros:
    - X: (n, p)
    - y: (n,), con valores 0 y 1
    - C: hiperparámetro de regularización

    Retorna:
    - beta: (p,)
    - beta0: float
    - lam: multiplicadores de las restricciones de margen (n,)
    """

    y = np.where(y == 1, 1.0, -1.0)  # (n,)
    n, p = X.shape
    beta, beta0, xi = cp.Variable(p), cp.Variable(), cp.Variable(n)
    margen = cp.multiply(y, X @ beta + beta0) >= 1 - xi  # restricción y_i (β^T x_i + β0) >= 1 - ξ_i
    problema = cp.Problem(cp.Minimize(0.5 * cp.sum_squares(beta) + C * cp.sum(xi)), [margen, xi >= 0])
    problema.solve(solver=cp.CLARABEL, tol_gap_abs=1e-10, tol_gap_rel=1e-10, tol_feas=1e-10)
    return beta.value, beta0.value, margen.dual_value

##### Comparación primal y dual con datos simulados

In [ ]:
rng = np.random.default_rng(0)
X_sim = np.vstack([rng.normal([-1, -1], 1.0, size=(50, 2)), rng.normal([1, 1], 1.0, size=(50, 2))])  # (100, 2)
y_sim = np.repeat([0, 1], 50)

dual = mySVM(C=1.0).fit(X_sim, y_sim)
beta_p, beta0_p, lam_p = svm_primal(X_sim, y_sim, C=1.0)

print("Dual: β =", dual.beta.round(5), "β0 =", round(dual.beta0, 5))
print("Primal: β =", beta_p.round(5), "β0 =", round(float(beta0_p), 5))
print(f"Máx. diferencia entre los λ del dual y los multiplicadores del primal: {np.abs(dual.lam - lam_p).max():.1e}")
print(f"Vectores soporte: {dual.es_sv.sum()} de {len(y_sim)}")

---

## Ejercicio 2

Los datos están en `datos/heart.csv`: las 13 variables del dataset y la columna `enfermedad` (1 si hay enfermedad cardíaca, 0 si no). Usamos solo `age` y `thalach`, estandarizadas con la media y el desvío de train.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

heart = pd.read_csv("datos/heart.csv")
X = heart[["age", "thalach"]].to_numpy(dtype=float)  # (303, 2)
y = heart["enfermedad"].to_numpy()  # (303,)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=0,
    stratify=y,
)
escalador = StandardScaler().fit(X_train)
X_train = escalador.transform(X_train)
X_test = escalador.transform(X_test)
print("Train:", X_train.shape, "| Test:", X_test.shape, "| proporción con enfermedad:", y.mean().round(3))

Función auxiliar para graficar: regiones de decisión, frontera $f(x)=0$, márgenes $f(x)=\pm1$ y vectores soporte.

In [ ]:
def graficar_svm(modelo, X, y, ax, titulo=""):
    """
    Grafica un mySVM ajustado en R^2.

    Parámetros:
    - modelo: mySVM ajustado
    - X: datos de entrenamiento (n, 2)
    - y: etiquetas (n,), con valores 0 y 1
    - ax: eje de matplotlib
    - titulo: título del gráfico
    """
    x1 = np.linspace(X[:, 0].min() - 0.5, X[:, 0].max() + 0.5, 300)
    x2 = np.linspace(X[:, 1].min() - 0.5, X[:, 1].max() + 0.5, 300)
    xx1, xx2 = np.meshgrid(x1, x2)
    F = modelo.decision_function(np.column_stack([xx1.ravel(), xx2.ravel()])).reshape(xx1.shape)

    ax.contourf(xx1, xx2, (F >= 0).astype(int), levels=[-0.5, 0.5, 1.5], colors=["tab:blue", "tab:red"], alpha=0.12)
    ax.contour(xx1, xx2, F, levels=[-1, 0, 1], colors="k", linestyles=["--", "-", "--"], linewidths=[1, 1.5, 1])
    ax.scatter(*X[y == 0].T, c="tab:blue", s=12, label="sin enfermedad")
    ax.scatter(*X[y == 1].T, c="tab:red", s=12, label="con enfermedad")
    ax.scatter(*modelo.X_sv.T, s=50, facecolors="none", edgecolors="k", linewidths=0.6, label="vectores soporte")
    ax.set_title(titulo, fontsize=10)
    ax.set_xlabel("age (estandarizada)")
    ax.set_ylabel("thalach (estandarizada)")

#### a) Kernel lineal: efecto de $C$

In [ ]:
valores_C = [0.01, 0.1, 1, 10, 100]
filas = []
fig, axes = plt.subplots(1, len(valores_C), figsize=(22, 4.5), sharey=True)
for ax, C in zip(axes, valores_C):
    modelo = mySVM(C=C, kernel="linear").fit(X_train, y_train)
    filas.append({
        "C": C,
        "accuracy test": np.mean(modelo.predict(X_test) == y_test),
        "vectores soporte": modelo.es_sv.sum(),
        "sobre el margen": modelo.sobre_margen.sum(),
        "en la cota (λ = C)": (modelo.es_sv & ~modelo.sobre_margen).sum(),
        "||β||": np.linalg.norm(modelo.beta),
    })
    graficar_svm(modelo, X_train, y_train, ax, f"lineal, C = {C}")
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

pd.DataFrame(filas).round(3)

**Para discutir:** ¿Cómo cambian el margen y la cantidad de vectores soporte al aumentar $C$?

#### b) Kernel RBF: efecto de $C$ y $\gamma$

In [ ]:
valores_C = [0.1, 1, 10, 100]
valores_gamma = [0.1, 1, 10]

filas = []
for gamma in valores_gamma:
    for C in valores_C:
        modelo = mySVM(C=C, kernel="rbf", gamma=gamma).fit(X_train, y_train)
        filas.append({
            "γ": gamma,
            "C": C,
            "accuracy train": np.mean(modelo.predict(X_train) == y_train),
            "accuracy test": np.mean(modelo.predict(X_test) == y_test),
            "vectores soporte": modelo.es_sv.sum(),
        })
resultados_rbf = pd.DataFrame(filas)
resultados_rbf.pivot_table(index="γ", columns="C", values=["accuracy test", "vectores soporte"]).round(3)

In [ ]:
combinaciones = [(1, 0.1), (1, 1), (1, 10), (0.1, 1), (10, 1), (100, 10)]  # (C, γ)
fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=True, sharey=True)
for ax, (C, gamma) in zip(axes.ravel(), combinaciones):
    modelo = mySVM(C=C, kernel="rbf", gamma=gamma).fit(X_train, y_train)
    acc = np.mean(modelo.predict(X_test) == y_test)
    graficar_svm(modelo, X_train, y_train, ax, f"RBF, C = {C}, γ = {gamma} (test {acc:.2f})")
plt.tight_layout()
plt.show()

**Para discutir:** ¿Mejora el kernel RBF al lineal con estas dos variables? ¿Qué pasa con la frontera cuando $\gamma$ es grande?

#### c) Comparación con `SVC` de `scikit-learn`

In [ ]:
from sklearn.svm import SVC

propio = mySVM(C=1, kernel="linear").fit(X_train, y_train)
sk = SVC(C=1, kernel="linear", tol=1e-8).fit(X_train, y_train)

print("β propio:", propio.beta.round(5), "| β sklearn:", sk.coef_[0].round(5))
print("β0 propio:", round(propio.beta0, 5), "| β0 sklearn:", round(sk.intercept_[0], 5))
print("Vectores soporte: propio", propio.es_sv.sum(), "| sklearn", sk.support_.size)
print("¿Son las mismas observaciones?", np.array_equal(np.where(propio.es_sv)[0], np.sort(sk.support_)))
print(f"Accuracy en test: propio {np.mean(propio.predict(X_test) == y_test):.4f}, sklearn {sk.score(X_test, y_test):.4f}")

# sklearn guarda λ_i y_i de los vectores soporte en dual_coef_
lam_sk = np.zeros(len(y_train))
lam_sk[sk.support_] = np.abs(sk.dual_coef_[0])
print(f"Máx. diferencia entre los λ: {np.abs(propio.lam - lam_sk).max():.1e}")

# Valor del objetivo dual con cada λ
y_pm = np.where(y_train == 1, 1.0, -1.0)
P = y_pm[:, None] * y_pm[None, :] * (X_train @ X_train.T)
objetivo_dual = lambda lam: 0.5 * lam @ P @ lam - lam.sum()
print(f"Objetivo dual: propio {objetivo_dual(propio.lam):.6f}, sklearn {objetivo_dual(lam_sk):.6f}")

#### d) Análisis de los vectores soporte del mejor modelo

El kernel lineal con $C\ge0.1$ tiene el mayor accuracy en test (0.703), pero ya lo analizamos en el ítem a; acá tomamos el mejor modelo RBF, que tiene más vectores soporte sobre el margen. (Elegir con los datos de test hace que ese accuracy sea optimista; lo correcto sería elegir por validación cruzada sobre train, como en la guía 2.4.)

In [ ]:
mejor = resultados_rbf.loc[resultados_rbf["accuracy test"].idxmax()]
C_mejor, gamma_mejor = mejor["C"], mejor["γ"]
modelo = mySVM(C=C_mejor, kernel="rbf", gamma=gamma_mejor).fit(X_train, y_train)
print(f"Mejor modelo: RBF, C = {C_mejor}, γ = {gamma_mejor}, accuracy en test = {mejor['accuracy test']:.3f}")

y_pm = np.where(y_train == 1, 1, -1)
margen_funcional = y_pm * modelo.decision_function(X_train)  # y_i f(x_i), (n,)
en_cota = modelo.es_sv & ~modelo.sobre_margen

pd.DataFrame(
    {
        "cantidad": [
            (~modelo.es_sv).sum(),
            modelo.sobre_margen.sum(),
            (en_cota & (margen_funcional >= 0)).sum(),
            (en_cota & (margen_funcional < 0)).sum(),
        ],
        "y_i f(x_i)": [
            f"min {margen_funcional[~modelo.es_sv].min():.3f}",
            f"entre {margen_funcional[modelo.sobre_margen].min():.4f} y {margen_funcional[modelo.sobre_margen].max():.4f}",
            "entre 0 y 1",
            "< 0",
        ],
    },
    index=[
        "no son vectores soporte (λ = 0)",
        "sobre el margen (0 < λ < C)",
        "en la cota (λ = C), bien clasificados dentro del margen",
        "en la cota (λ = C), mal clasificados",
    ],
)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
graficar_svm(modelo, X_train, y_train, ax, f"RBF, C = {C_mejor}, γ = {gamma_mejor}")
ax.scatter(*X_train[modelo.sobre_margen].T, marker="s", s=90, facecolors="none", edgecolors="green", linewidths=1.5, label="sobre el margen")
ax.legend(fontsize=8)
plt.show()

# Cantidad de vectores soporte en función de C, con el γ del mejor modelo
for C in [0.1, 1, 10, 100]:
    print(f"C = {C}: {mySVM(C=C, kernel='rbf', gamma=gamma_mejor).fit(X_train, y_train).es_sv.sum()} vectores soporte")